# 01 · Data: boreholes → 0.5 m cells (V3 rules)

1. Reads the borehole file and keeps the **Downtown** boreholes (V3 = Downtown only).
2. Cuts every borehole into 0.5 m cells (0-15 m) and labels them **Clay = 1** (CL, CH) or **Sand = 0** (SC, SM, SP).
3. Fills gaps ≤ 3 m between two cells of the same class when no record lies inside the gap (V3 rule).
4. Marks the **11 external-holdout boreholes** chosen by V3 before any modelling (read from the V3 folder and checked against V3's lock).
5. Checks that the cells are **identical to V3's**, then saves them.

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])

## 1. Cells

In [ ]:
records = H.load_downtown(S)
cells, info_cells = H.make_cells(records, S)
cells, info_fill = H.fill_gaps(cells, records, S)
holdout = H.load_v3_holdout(S)
cells["set"] = np.where(cells.BH_Code.isin(holdout.index), "external", "development")
print("Labelling:", info_cells)
print("Gap filling:", info_fill)
t = cells.groupby("set").agg(boreholes=("BH_Code", "nunique"), cells=("y", "size"), measured=("measured", "sum"),
                             clay_share=("y", "mean"))
t["filled"] = t.cells - t.measured
display(t.round(3))

## 2. Save, and check against V3
The cells are saved to CSV and read back, exactly as V3 did (reading a CSV can change the last digit of a number, so this step matters for identical results).

In [ ]:
P = H.processed_dir(S)
cells.to_csv(P / "cells.csv", index=False)
cells = pd.read_csv(P / "cells.csv")
v3 = pd.read_csv(S["v3_root"] / "data" / "processed" / "cells_all.csv")
v3 = v3[v3.is_downtown & v3.set.isin(["development", "external"])].reset_index(drop=True)
cols = ["BH_Code", "cell", "depth_top_m", "UTM_Easting_m", "UTM_Northing_m", "Top_Elevation_m", "y", "measured", "set"]
same = len(cells) == len(v3) and all((cells[c].values == v3[c].values).all() for c in cols)
print("Cells identical to V3 (values and order):", same)
assert same, "the cells differ from V3: stop and check the data files"

## 3. Clay share by depth

In [ ]:
dev = cells[cells.set == "development"]
prof = pd.DataFrame({"measured cells": dev[dev.measured == 1].groupby("depth_top_m").y.mean(),
                     "all cells": dev.groupby("depth_top_m").y.mean()})
prof.to_csv(H.out_dir(S, "01_data") / "clay_share_by_depth.csv")
summary = {"boreholes": int(cells.BH_Code.nunique()), "development_boreholes": int(dev.BH_Code.nunique()),
           "holdout_boreholes": int(len(holdout)), "cells": len(cells), "development_cells": len(dev),
           "development_filled_cells": int((dev.measured == 0).sum()), "labelling": info_cells, "gap_filling": info_fill,
           "identical_to_v3": bool(same)}
(H.out_dir(S, "01_data") / "data_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 02.")